In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chưa bật GPU")

CUDA available: True
GPU: Tesla T4


In [2]:
!pip uninstall -y transformers tokenizers FlagEmbedding
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "FlagEmbedding==1.2.11" accelerate sentence-transformers

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.1/147.1 kB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content

!rm -rf Text-Mining---RAG-on-News
!git clone -b Alibaba https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git

%cd /content/Text-Mining---RAG-on-News

/content
Cloning into 'Text-Mining---RAG-on-News'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 437 (delta 40), reused 60 (delta 15), pack-reused 344 (from 1)
Receiving objects: 100% (437/437), 21.11 MiB | 14.44 MiB/s, done.
Resolving deltas: 100% (214/214), done.
Updating files: 100% (62/62), done.
/content/Text-Mining---RAG-on-News


In [5]:
import torch
import transformers
import importlib.util

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("transformers:", transformers.__version__)
print("FlagEmbedding installed:", importlib.util.find_spec("FlagEmbedding") is not None)

CUDA available: True
GPU: Tesla T4
transformers: 4.44.2
FlagEmbedding installed: True


In [6]:
from pathlib import Path

input_path = Path("/content/drive/MyDrive/out_embedding/per_query_structured.jsonl")

print("Input exists:", input_path.exists())
print("Input path:", input_path)

if input_path.exists():
    with input_path.open("r", encoding="utf-8") as f:
        first_line = f.readline()
    print(first_line[:500])

Input exists: True
Input path: /content/drive/MyDrive/out_embedding/per_query_structured.jsonl
{"qa_id": "211640_1", "qa_type": "factoid", "question": "Những loại nội tạng động vật nào được khuyến cáo nên hạn chế để tránh tăng axit uric và hại thận?", "gold_articles": ["211640"], "top_articles": ["211640", "211640", "28856", "28856", "207094", "205401", "28348", "31222", "27887", "27887"], "candidates": [{"rank": 1, "chunk_index": 1, "chunk_id": "211640_structured_0001", "article_id": "211640", "score": 0.884665, "text": "Tiêu đề: Không muốn hại thận, cần hạn chế 4 loại thịt\nMô tả: Purin


In [7]:
from pathlib import Path

script_path = Path("src/re-ranker/bge_rerank_structure.py")

print("Script exists:", script_path.exists())
print("Script path:", script_path)

Script exists: True
Script path: src/re-ranker/bge_rerank_structure.py


In [8]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/out_reranker")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output dir exists:", output_dir.exists())

Output dir exists: True


In [9]:
from huggingface_hub import login
login()

In [ ]:
!python src/re-ranker/bge_rerank_token.py \
    --input "/content/drive/MyDrive/out_embedding/per_query_structured.jsonl" \
    --output "/content/drive/MyDrive/out_reranker/rerank_structure_bge_top5_test.jsonl" \
    --limit 5 \
    --batch-size 8 \
    --fp16

2026-07-06 08:57:00.388303: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Done. Wrote 5 rows to /content/drive/MyDrive/out_reranker/rerank_token_bge_top5_test.jsonl


In [ ]:
import json
from pathlib import Path

test_output = Path("/content/drive/MyDrive/out_reranker/rerank_token_bge_top5_test.jsonl")

print("Test output exists:", test_output.exists())

with test_output.open("r", encoding="utf-8") as f:
    row = json.loads(next(f))

print("QA ID:", row["qa_id"])
print("Question:", row["question"])
print("Embedding type:", row["embedding_type"])
print("Reranker:", row["reranker"])
print("Metrics:", row["rerank_metrics"])
print("Top 1 score:", row["reranked_candidates"][0]["rerank_score"])
print("Top 1 text:", row["reranked_candidates"][0]["text"][:300])

Test output exists: True
QA ID: 211640_1
Question: Những loại nội tạng động vật nào được khuyến cáo nên hạn chế để tránh tăng axit uric và hại thận?
Embedding type: token
Reranker: BAAI/bge-reranker-v2-m3
Metrics: {'hit@1': 1.0, 'hit@5': 1.0, 'recall@5': 1.0, 'mrr@5': 1.0, 'ndcg@5': 1.6309297535714575}
Top 1 score: 4.625
Top 1 text: Tiêu đề: Không muốn hại thận, cần hạn chế 4 loại thịt
Mô tả: Purin trong các loại thịt đỏ chuyển hóa thành axit uric khiến thận phải tăng cường hoạt động. Nếu không xử lý kịp, thận sẽ bị ảnh hưởng nặng nề.
Chuyên mục: Sức khỏe
Đoạn nội dung:
Theo Sohu, nhiều người khi nhắc đến tăng axit uric, gout h


In [10]:
!python src/re-ranker/bge_rerank_token.py \
    --input "/content/drive/MyDrive/out_embedding/per_query_structured.jsonl" \
    --output "/content/drive/MyDrive/out_reranker/rerank_structure_bge_top5.jsonl" \
    --batch-size 8 \
    --fp16

2026-07-07 11:19:45.536916: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 1.17kB [00:00, 2.26MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 8.85MB/s]
tokenizer.json: 100% 17.1M/17.1M [00:00<00:00, 44.9MB/s]
special_tokens_map.json: 100% 964/964 [00:00<00:00, 7.00MB/s]
config.json: 100% 795/795 [00:00<00:00, 6.13MB/s]
model.safetensors: 100% 2.27G/2.27G [00:28<00:00, 80.1MB/s]
Reranked 10 queries
Reranked 20 queries
Reranked 30 queries
Reranked 40 queries
Reranked 50 queries
Reranked 60 queries
Reranked 70 queries
Reranked 80 queries
Reranked 90 queries
Reranked 100 queries
Reranked 110 queries
Done. Wrote 114 rows to /content/drive/MyDrive/out_reranker/rerank_structure_bge_top5.jsonl


In [11]:
from pathlib import Path

input_file = Path("/content/drive/MyDrive/out_embedding/per_query_structured.jsonl")
output_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_bge_top5.jsonl")

def count_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

print("Input lines:", count_jsonl(input_file))
print("Output lines:", count_jsonl(output_file))

Input lines: 114
Output lines: 114


In [13]:
import json
import pandas as pd
from pathlib import Path

output_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_bge_top5.jsonl")
summary_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_bge_top5_summary.csv")

rows = []

with output_file.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        rows.append(row["rerank_metrics"])

df = pd.DataFrame(rows)

summary = {
    "config": "token_bge_reranker",
    "num_queries": len(df),
    "hit@1": df["hit@1"].mean(),
    "hit@5": df["hit@5"].mean(),
    "recall@5": df["recall@5"].mean(),
    "mrr@5": df["mrr@5"].mean(),
    "ndcg@5": df["ndcg@5"].mean(),
}

summary_df = pd.DataFrame([summary])


summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")
print(f"Saved summary to {summary_file}")
summary_df

Saved summary to /content/drive/MyDrive/out_reranker/rerank_structure_bge_top5_summary.csv


,config,num_queries,hit@1,hit@5,recall@5,mrr@5,ndcg@5
0,token_bge_reranker,114,0.649123,0.684211,0.684211,0.666667,0.671261
